# Simulation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/notebooks/intro/05_simulation.ipynb)

Official API intro to continuous-time simulation: façades vs `Simulator`, time
grids, and solver modes. Discrete-in-the-loop (`StepSystem`, `Computer`, hybrid)
is covered in [`intro/06_hybrid.ipynb`](06_hybrid.ipynb).

**Scripts / tooling:** `examples/scripts/` · [`tooling/benchmark.ipynb`](../tooling/benchmark.ipynb)


In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")


## Façade: `compute_trajectory`

Most teaching demos call the façade on a `System` — it compiles, integrates, and
caches a `Trajectory`.


In [ ]:
import numpy as np
from minilink.dynamics.catalog.pendulum.pendulum import Pendulum

plant = Pendulum()
plant.x0 = np.array([0.5, 0.0])
traj = plant.compute_trajectory(tf=5.0)
plant.plot_trajectory()
print(type(traj), getattr(traj, "t", None) is not None)


## `Simulator` and solver modes

For explicit control of solver, step size, and reporting, construct a `Simulator`.


## ODE solvers

Time integration lives in `minilink.simulation`. The central class is
`Simulator`: it compiles the model, builds a time grid, and integrates
`dx/dt = f(x, u, t)`. Most workflows call façades on `System` instead
(`compute_trajectory` / `compute_forced`), which accept `solver=...` and
`compile_backend=...`.

Common solver modes include `rk4_fixedsteps`, `euler`, and SciPy IVP wrappers. For a
solver × backend sweep, see [`tooling/benchmark.ipynb`](../tooling/benchmark.ipynb).


In [ ]:
import numpy as np
from minilink.dynamics.catalog.pendulum.pendulum import Pendulum
from minilink.simulation.simulator import Simulator

plant = Pendulum()
plant.x0 = np.array([0.5, 0.0])

sim = Simulator(plant, t0=0.0, tf=5.0, dt=0.01, solver="rk4_fixedsteps")
traj = sim.solve()
print("t shape:", np.shape(traj.t), "x shape:", np.shape(traj.x))

# Same integration via the façade:
plant.compute_trajectory(tf=5.0, dt=0.01, solver="rk4_fixedsteps")
plant.plot_trajectory()
